# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.7 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID='task084'; CH=10; H=W=30
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'
ROOT=Path(COMPETITION) if Path(COMPETITION).exists() else Path('/mnt/data')
TASK_JSON=ROOT/'task084.json'
OUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data/task084_zero_pad')
OUT_DIR.mkdir(exist_ok=True)
ONNX_PATH=OUT_DIR/f'{TASK_ID}.onnx'
ZIP_PATH=OUT_DIR/'task084_zero_pad_submission.zip'
STATIC_ZIP=OUT_DIR/'task084_zero_pad_static_graph_submission.zip'
GENERIC_ZIP=OUT_DIR/'submission.zip'
AUDIT_JSON=OUT_DIR/'task084_zero_pad_audit.json'
AUDIT_CSV=OUT_DIR/'task084_zero_pad_audit.csv'

In [6]:
def make_masks():
    # Orientation k = np.rot90(canonical-left-rule, k CCW): 0 left, 1 bottom, 2 right, 3 top.
    side=[]; diag2=[]; side4=[]
    for k in range(4):
        side_k=[]; diag_k=[]; side4_k=[]
        for n in range(1,31):
            c=np.zeros((30,30), np.float32)
            d=np.zeros((30,30), np.float32)
            f=np.zeros((30,30), np.float32)
            c[:n,0]=1.0
            if n>=2:
                f[n-1,1:n]=1.0
                for r in range(n-1):
                    col=n-1-r
                    if 1 <= col < n:
                        d[r,col]=1.0
            cc=np.zeros_like(c); dd=np.zeros_like(d); ff=np.zeros_like(f)
            cc[:n,:n]=np.rot90(c[:n,:n], k=k)
            dd[:n,:n]=np.rot90(d[:n,:n], k=k)
            ff[:n,:n]=np.rot90(f[:n,:n], k=k)
            side_k.append(cc); diag_k.append(dd); side4_k.append(ff)
        side.append(np.stack(side_k)); diag2.append(np.stack(diag_k)); side4.append(np.stack(side4_k))
    return (torch.tensor(np.stack(side), dtype=torch.float32),
            torch.tensor(np.stack(diag2), dtype=torch.float32),
            torch.tensor(np.stack(side4), dtype=torch.float32))

class Task084ZeroPadD4Model(nn.Module):
    """D4-rotation-aware task084 tensor program with Kaggle zero-vector padding."""
    def __init__(self):
        super().__init__()
        side, diag2, side4 = make_masks()
        self.register_buffer('side_masks', side)      # [4,30,30,30]
        self.register_buffer('diag2_masks', diag2)    # [4,30,30,30]
        self.register_buffer('side4_masks', side4)    # [4,30,30,30]
        self.register_buffer('n_values', torch.arange(1,31).view(1,30,1,1).float())
    def _eq_count(self, count, n):
        return 1.0 - torch.clamp(torch.abs(count - n), 0.0, 1.0)
    def forward(self, x):
        # x is one-hot inside the real ARC grid, and all-zero outside padding.
        real = torch.clamp(x.sum(dim=1, keepdim=True), 0.0, 1.0)
        nonzero = x[:,1:,:,:]
        active = torch.clamp(nonzero.sum(dim=1, keepdim=True), 0.0, 1.0)  # non-background support only
        total = active.sum(dim=(2,3), keepdim=True)
        channels = [active*0.0 for _ in range(9)]
        for k in range(4):
            side = self.side_masks[k:k+1]  # [1,30,30,30]
            side_score = (active * side).sum(dim=(2,3), keepdim=True)
            sel = self._eq_count(total, self.n_values) * self._eq_count(side_score, self.n_values)
            side_union = (sel * side).sum(dim=1, keepdim=True)
            line_colored = nonzero * side_union
            for c in range(9):
                channels[c] = channels[c] + line_colored[:,c:c+1]
            diag = (sel * self.diag2_masks[k:k+1]).sum(dim=1, keepdim=True)
            base = (sel * self.side4_masks[k:k+1]).sum(dim=1, keepdim=True)
            channels[1] = channels[1] + diag  # color 2
            channels[3] = channels[3] + base  # color 4
        colored = torch.clamp(torch.cat(channels, dim=1) * real, 0.0, 1.0)
        nz = torch.clamp(colored.sum(dim=1, keepdim=True), 0.0, 1.0)
        bg = real * (1.0 - nz)
        return torch.cat([bg, colored], dim=1)

In [7]:
def grid_to_tensor(grid):
    g=np.array(grid,dtype=np.int64)
    x=np.zeros((1,10,30,30),dtype=np.float32)
    h,w=g.shape
    for r in range(h):
        for c in range(w):
            v=int(g[r,c])
            if 0 <= v < 10:
                x[0,v,r,c]=1.0
    return x

def grid_to_target_tensor(grid):
    return grid_to_tensor(grid)

def pad_grid(grid):
    a=np.array(grid,dtype=np.int64)
    p=np.zeros((30,30),dtype=np.int64)
    p[:a.shape[0],:a.shape[1]]=a
    return p

def exact_eval(sess, examples):
    name=sess.get_inputs()[0].name
    exact=0; first=None
    for i,ex in enumerate(examples):
        y=sess.run(None,{name:grid_to_tensor(ex['input'])})[0]
        target_t=grid_to_target_tensor(ex['output'])
        ok=np.array_equal(y, target_t)
        exact += int(ok)
        if not ok and first is None:
            pred=y[0].argmax(axis=0).astype(np.int64)
            target=pad_grid(ex['output'])
            first={'idx':i,'raw_l1':float(np.abs(y-target_t).sum()),'argmax_wrong_pixels':int((pred!=target).sum()),
                   'unique_values':sorted(np.unique(y).tolist()),
                   'sum_min':float(y[0].sum(axis=0).min()),'sum_max':float(y[0].sum(axis=0).max())}
    return {'exact':exact,'total':len(examples),'first_wrong':first}

def rotate_example(ex,k):
    inp=np.array(ex['input'],dtype=np.int64); out=np.array(ex['output'],dtype=np.int64)
    return {'input':np.rot90(inp,k=k).tolist(), 'output':np.rot90(out,k=k).tolist()}

def synth_example(n,color,k):
    inp=np.zeros((n,n),dtype=np.int64); out=np.zeros((n,n),dtype=np.int64)
    inp[:,0]=color
    out[:,0]=color
    if n>=2:
        out[n-1,1:n]=4
        for r in range(n-1):
            c=n-1-r
            if 1<=c<n: out[r,c]=2
    if k:
        inp=np.rot90(inp,k=k); out=np.rot90(out,k=k)
    return {'input':inp.tolist(), 'output':out.tolist()}

def op_counts(m):
    return dict(collections.Counter(n.op_type for n in m.graph.node))

def onnx_shape(value_info):
    return [int(d.dim_value) if d.dim_value else None for d in value_info.type.tensor_type.shape.dim]

def raw_zero_padding_check(sess, examples):
    name=sess.get_inputs()[0].name
    checked=0
    for ex in examples[:min(50,len(examples))]:
        y=sess.run(None,{name:grid_to_tensor(ex['input'])})[0]
        vals=np.unique(y)
        sums=y[0].sum(axis=0)
        h,w=np.array(ex['output']).shape
        inside=sums[:h,:w]; outside=sums.copy(); outside[:h,:w]=0
        if not np.all(np.isin(vals,[0.0,1.0])) or not np.allclose(inside,1.0) or not np.allclose(outside,0.0):
            return {'ok':False,'unique_values':vals.tolist(),'inside_min':float(inside.min()),'inside_max':float(inside.max()),'outside_max':float(outside.max())}
        checked += 1
    return {'ok':True,'checked':checked}

def export_and_audit():
    task=json.load(open(TASK_JSON))
    model=Task084ZeroPadD4Model().eval()
    dummy=torch.zeros(1,10,30,30,dtype=torch.float32)
    torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=13,dynamic_axes=None,do_constant_folding=True,dynamo=False)
    m=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(m)
    sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])
    ops=op_counts(m)
    forbidden={'Loop','Scan','NonZero','Unique','Script','Function'}
    risky={'Shape','Gather','ConstantOfShape','Expand','Range','ScatterND'}
    rotated=[]
    for split in ['train','test','arc-gen']:
        for ex in task.get(split,[]):
            for k in range(4):
                rotated.append(rotate_example(ex,k))
    synth=[]
    for n in range(3,31):
        for color in range(1,10):
            for k in range(4):
                synth.append(synth_example(n,color,k))
    ag=task['arc-gen']; cut=int(round(len(ag)*0.40)); holdout=ag[cut:]
    audit={
        'task':TASK_ID,
        'input_shape':onnx_shape(m.graph.input[0]),
        'output_shape':onnx_shape(m.graph.output[0]),
        'onnx_size_bytes':ONNX_PATH.stat().st_size,
        'ops':ops,
        'forbidden_ops_present':{op:ops.get(op,0) for op in forbidden if ops.get(op,0)},
        'risky_ops_present':{op:ops.get(op,0) for op in risky if ops.get(op,0)},
        'train':exact_eval(sess, task['train']),
        'test':exact_eval(sess, task['test']),
        'arc_gen_all':exact_eval(sess, ag),
        'arc_gen_holdout_60pct':exact_eval(sess, holdout),
        'rotated_train_test_arcgen':exact_eval(sess, rotated),
        'synthetic_d4': exact_eval(sess, synth),
        'raw_zero_padding': raw_zero_padding_check(sess, task['train']+task['test']+holdout),
    }
    with open(AUDIT_JSON,'w') as f: json.dump(audit,f,indent=2)
    with open(AUDIT_CSV,'w',newline='') as f:
        writer=csv.writer(f); writer.writerow(['metric','value'])
        for k in ['input_shape','output_shape','onnx_size_bytes','forbidden_ops_present','risky_ops_present','train','test','arc_gen_all','arc_gen_holdout_60pct','rotated_train_test_arcgen','synthetic_d4','raw_zero_padding']:
            writer.writerow([k,json.dumps(audit[k])])
    for zp in [ZIP_PATH, STATIC_ZIP, GENERIC_ZIP]:
        with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED) as z:
            z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
    return audit


In [8]:
audit=export_and_audit()
print(json.dumps(audit, indent=2)[:5000])

/tmp/ipykernel_16/2157910590.py:78: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=13,dynamic_axes=None,do_constant_folding=True,dynamo=False)


{
  "task": "task084",
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 1324916,
  "ops": {
    "Constant": 175,
    "ReduceSum": 20,
    "Clip": 9,
    "Slice": 37,
    "Mul": 27,
    "Sub": 11,
    "Abs": 5,
    "Add": 44,
    "Concat": 2
  },
  "forbidden_ops_present": {},
  "risky_ops_present": {},
  "train": {
    "exact": 3,
    "total": 3,
    "first_wrong": null
  },
  "test": {
    "exact": 1,
    "total": 1,
    "first_wrong": null
  },
  "arc_gen_all": {
    "exact": 171,
    "total": 171,
    "first_wrong": null
  },
  "arc_gen_holdout_60pct": {
    "exact": 103,
    "total": 103,
    "first_wrong": null
  },
  "rotated_train_test_arcgen": {
    "exact": 700,
    "total": 700,
    "first_wrong": null
  },
  "synthetic_d4": {
    "exact": 1008,
    "total": 1008,
    "first_wrong": null
  },
  "raw_zero_padding": {
    "ok": true,
    "checked": 50
  }
}
